In [1]:
#@title 1. Install package
#@markdown # 1. Install package
#@markdown Github Project: https://github.com/Sanster/lama-cleaner

#!pip3 install lama-cleaner pyngrok

In [2]:
#@title 2. Download model
#@markdown # 2. Downloading model
init_model = 'lama' #@param ['lama', 'sd1.5', 'paint_by_example', 'ldm']
port = 4242

#@markdown # !! Important Notes !!
#@markdown Please stop this block after model download finish (seeing `running on http://0.0.0.0:4242/` in the log)

!lama-cleaner --host 0.0.0.0 --port $port --model $init_model

- Platform: Linux-5.15.0-118-generic-x86_64-with-glibc2.31
- Python version: 3.10.13
- torch: 2.4.0+cu121
- torchvision: 0.19.0+cu121
- Pillow: 9.5.0
- diffusers: 0.27.2
- transformers: 4.44.0
- opencv-python: 4.8.0.76
- xformers: 0.0.27.post2
- accelerate: 0.26.1
- lama-cleaner: 1.2.5
- rembg: N/A
- realesrgan: N/A
- gfpgan: N/A

/usr/local/lib/python3.10/site-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/usr/local/lib/python3.10/site-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")
/usr/local/lib/py

In [3]:
ngrok_authtoken = '2kbRU3Z4sQ8D9udjvxy6aFkdEfl_2zfFC4Y7HsjZUCGPfAUYe' #@param {type: 'string'}

!ngrok authtoken $ngrok_authtoken

from pyngrok import ngrok
public_url = ngrok.connect(port).public_url
print(f"ngrok public url: {public_url}.")

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


t=2024-08-22T17:45:15+0000 lvl=warn msg="can't bind default web address, trying alternatives" obj=web addr=127.0.0.1:4040
t=2024-08-22T17:45:15+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Your account is limited to 1 simultaneous ngrok agent sessions.\nYou can run multiple simultaneous tunnels from a single agent session by defining the tunnels in your agent configuration file and starting them with the command `ngrok start --all`.\nRead more about the agent configuration file: https://ngrok.com/docs/secure-tunnels/ngrok-agent/reference/config\nYou can view your current agent sessions in the dashboard:\nhttps://dashboard.ngrok.com/tunnels/agents\r\n\r\nERR_NGROK_108\r\n"

ERROR:  authentication failed: Your account is limited to 1 simultaneous ngrok agent sessions.
ERROR:  You can run multiple simultaneous tunnels from a single agent session by defining the tunnels in your agent configuration file and starting them with the command `n

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Your account is limited to 1 simultaneous ngrok agent sessions.\nYou can run multiple simultaneous tunnels from a single agent session by defining the tunnels in your agent configuration file and starting them with the command `ngrok start --all`.\nRead more about the agent configuration file: https://ngrok.com/docs/secure-tunnels/ngrok-agent/reference/config\nYou can view your current agent sessions in the dashboard:\nhttps://dashboard.ngrok.com/tunnels/agents\r\n\r\nERR_NGROK_108\r\n.

In [ ]:
#@title 4. Start Lama Cleaner server
#@markdown # 4. Start Lama Cleaner server
#@markdown When you see `Running on http://0.0.0.0:4242/' in the log`, please open **ngrok public link**

# lama-cleaner --host 0.0.0.0 --port 4242 --model 'lama'
# lama-cleaner --host 0.0.0.0 --port 4242 --model 'ldm'
# lama-cleaner --host 0.0.0.0 --port 4242 --model 'sd1.5'


#!lama-cleaner --host 0.0.0.0 --port $port --model $init_model

In [4]:
from diffusers.utils import load_image
from diffusers import AutoPipelineForInpainting
import torch
import os, sys
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from tqdm import tqdm
import requests
from io import BytesIO

# Add the project root directory to sys.path
project_root = os.path.abspath('..')  # Adjust this path if needed
sys.path.append(project_root)

from utils import load_images_and_combine
from utils import mask_on_image, dilate_mask

2024-08-22 17:45:50.710109: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-22 17:45:50.725351: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-22 17:45:50.729989: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-08-22 17:45:50.744388: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-22 17:45:52.043891: W tensorflow/compiler/tf2

[08/22/24 17:46:08] WARNING  Your inference package version 0.12.1 is out of date! Please upgrade to ]8;id=335154;file:///usr/local/lib/python3.10/site-packages/inference/core/__init__.py\__init__.py]8;;\:]8;id=186271;file:///usr/local/lib/python3.10/site-packages/inference/core/__init__.py#35\35]8;;\
                             version 0.16.2 of inference for the latest features and bug fixes by                  
                             running `pip install --upgrade inference`.                                            

In [5]:
url = f'http://127.0.0.1:{port}/inpaint'

form_data = dict([
    ('ldmSteps', '25'), ('ldmSampler', 'plms'), ('zitsWireframe', 'true'),
    ('hdStrategy', 'Crop'), ('hdStrategyCropMargin', '196'), ('hdStrategyCropTrigerSize', '800'),
    ('hdStrategyResizeLimit', '2048'), ('prompt', ''), ('negativePrompt', ''),
    ('croperX', '180'), ('croperY', '443'), ('croperHeight', '512'), ('croperWidth', '512'),
    ('useCroper', 'false'), ('sdMaskBlur', '5'), ('sdStrength', '0.75'), ('sdSteps', '50'),
    ('sdGuidanceScale', '7.5'), ('sdSampler', 'uni_pc'), ('sdSeed', '-1'), ('sdMatchHistograms', 'false'),
    ('sdScale', '1'), ('cv2Radius', '5'), ('cv2Flag', 'INPAINT_NS'), ('paintByExampleSteps', '50'),
    ('paintByExampleGuidanceScale', '7.5'), ('paintByExampleSeed', '-1'), ('paintByExampleMaskBlur', '5'),
    ('paintByExampleMatchHistograms', 'false'), ('p2pSteps', '50'), ('p2pImageGuidanceScale', '1.5'),
    ('p2pGuidanceScale', '7.5'), ('controlnet_conditioning_scale', '0.4'),
    ('controlnet_method', 'control_v11p_sd15_canny')
    ])


In [6]:
def plot_grid(figures, num_cols=3, with_token=False, save_path=None):
    # Define the number of columns for the grid
    num_rows = (len(figures) + num_cols - 1) // num_cols  # Calculate the number of rows needed
    
    # Create a figure and a set of subplots
    if num_cols>4: 
        fig, axs = plt.subplots(num_rows, num_cols, figsize=(15, 2 * num_rows))
    else:
        fig, axs = plt.subplots(num_rows, num_cols, figsize=(15, 5 * num_rows))

    # Flatten the axes array for easy iteration
    axs = axs.flatten()

    # Plot each figure in the grid
    for i, fig in enumerate(figures):
        if with_token is True:
            axs[i].imshow(fig[0]) 
            axs[i].axis('off')      # Turn off the axis
            axs[i].set_title(fig[1])    # add title to subplot
        else:
            axs[i].imshow(fig)  
            axs[i].axis('off')      # Turn off the axis


    # Turn off any remaining empty subplots
    for j in range(i + 1, len(axs)):
        axs[j].axis('off')

    if save_path is not None:
        plt.savefig(save_path)

    # Show the grid of figures
    plt.tight_layout()
    plt.show()

In [7]:
datasets = {
    'ISIC2019': {
        'dataset_dir': 'ISIC2019',
        'confounder_dir': "patch",
        'no_confounder_dir': "no_patch",
        'custom_vocabulary': ['colorful patch', 'colorful sticker', 'colorful circle', 'colorful cutted circle'],
        #["colorful cutted circle", "colorful circle", "coloful sticker", "colorful patch", "part of colorful sticker", "part of colorful patch", "part of colorful circle", "colorful semicircle", "colorful ellipsoid", "coloful cutted ellipsoid"]
        'confidence_threshold': 0.0007
    },

    'Pascal_VOC07': {
        'dataset_dir': 'Pascal_VOC07',
        'confounder_dir': 'horse_watermark',
        'no_confounder_dir': 'horse_no_watermark',
        'custom_vocabulary': ['watermark', 'signature', 'timestamp', 'letters'],
        'confidence_threshold': 0.0001  
    },

    'DecoyMNIST': {
        'dataset_dir': 'DecoyMNIST',
        'confounder_dir': 'confounder',
        'no_confounder_dir': 'no_confounder',
        'custom_vocabulary': ['square', 'confounder', 'rectangle'],
        'confidence_threshold': 0.0001
    }
}

In [8]:
# Load images
num_imgs = 100
dataset_key ='ISIC2019'# 'Pascal_VOC07' #
dataset = datasets[dataset_key]
images = load_images_and_combine(dir_path=f"/workspace/home/data/{dataset['dataset_dir']}/raw", confounder_dir=dataset['confounder_dir'], no_confounder_dir=dataset['no_confounder_dir'], num_imgs=num_imgs)
segmentation_masks = load_images_and_combine(dir_path=f"/workspace/home/data/{dataset['dataset_dir']}/seg", confounder_dir=dataset['confounder_dir'], no_confounder_dir=dataset['no_confounder_dir'], num_imgs=num_imgs)

In [9]:
device = "cuda:0"
pipe = AutoPipelineForInpainting.from_pretrained("diffusers/stable-diffusion-xl-1.0-inpainting-0.1", torch_dtype=torch.float16, variant="fp16")
pipe = pipe.to(device)
pipe.enable_attention_slicing() # Recommended if your computer has < 64 GB of RAM

generator = torch.Generator(device).manual_seed(0)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

The config attributes {'decay': 0.9999, 'inv_gamma': 1.0, 'min_decay': 0.0, 'optimization_step': 37000, 'power': 0.6666666666666666, 'update_after_step': 0, 'use_ema_warmup': False} were passed to UNet2DConditionModel, but are not expected and will be ignored. Please verify your config.json configuration file.


In [10]:
def run_instance(image, mask, prompt, save_path, iterations=2):
    image_size = image.size
    def run_lama_cleaner(image, mask):
        # Create a BytesIO object
        img_byte_arr = BytesIO()
        mask_byte_arr = BytesIO()

        # Save the image to the BytesIO object in a specific format (e.g., JPEG, PNG)
        image.save(img_byte_arr, format='JPEG')
        mask.save(mask_byte_arr, format='JPEG')

        # Get the byte data
        img_byte_data = img_byte_arr.getvalue()
        mask_byte_data = mask_byte_arr.getvalue()

        files = {
            'image': ('image.jpeg', img_byte_data, 'image/jpeg'),
            'mask': ('blob', mask_byte_data, 'image/jpeg'),
        }


        response = requests.post(url, data=form_data, files=files)

        # display the result
        result = Image.open(response.content)
        result = Image.open(BytesIO(response.content))
        
        return result
        

    def run_diffuser(image, mask, prompt):
        original_img_size = image.size
        # Run the diffuser
        result = pipe(
                    prompt=prompt,
                    image=image,
                    mask_image=mask,
                    guidance_scale=8.0,
                    num_inference_steps=25,  # steps between 15 and 30 work well for us
                    strength=0.99,  # make sure to use `strength` below 1.0
                    generator=generator,
                    #padding_mask_crop=32,
                    ).images[0]

        result = result.resize(original_img_size)
        return result


    mask_np = np.array(mask)
    
    # Perform the dilation operation
    dilated_mask_image = dilate_mask(mask_np, 5, iterations=4)

    # Convert the dilated images back to PIL images for display
    dilated_mask_image_pil = Image.fromarray(dilated_mask_image).resize(image_size)

    mask = dilated_mask_image_pil

    mask_on_img = mask_on_image(image, mask)

    # Run models
    result_lama_cleaner = run_lama_cleaner(image, mask)
    result_diffuser = run_diffuser(result_lama_cleaner, mask, prompt)
    result_diffuser2 = run_diffuser(result_diffuser, mask, prompt)

    # Display the results
    #plot_grid([image, mask_image, mask_on_img, result_lama_cleaner, result_diffuser, result_diffuser2], num_cols=3, save_path=save_path)

    # save the result
    result_diffuser2.save(save_path)




In [11]:
prompt = "part of skin in homogenous skin color"
# "part of skin"
#"remove the confounder"
# ""
#"remove colorful patch"
#"lesion on skin"

for i, (image_dict, mask_dict) in enumerate(tqdm(zip(images[:num_imgs], segmentation_masks[:num_imgs]))):
  image = image_dict['image']
  meta_img = image_dict['meta']
  img_url = meta_img['file_path']

  mask = mask_dict['image']
  meta_mask = mask_dict['meta']
  mask_url = meta_mask['file_path']

  image = load_image(img_url)
  mask_image = load_image(mask_url)


  output_dir = f"/workspace/home/data/{dataset['dataset_dir']}/inpainted/"
  #f"/workspace/home/outputfiles/{dataset['dataset_dir']}/inpainting/"
  os.makedirs(output_dir, exist_ok=True)
  file_name = meta_img['file_name'].split('.')[0]
  save_path = os.path.join(output_dir, f'{file_name}_[{prompt}].jpeg')

  run_instance(image, mask_image, prompt, save_path)

0it [00:01, ?it/s]


FileNotFoundError: [Errno 2] No such file or directory: b'CUDA out of memory'